# 01 Market Universe

Purpose: inspect the processed MOEX market universe, tradability flags, liquidity ranking and data quality.

Data source: `data/processed/market_universe.parquet` generated from public/delayed MOEX ISS data and raw cache where available.


## Methodology Summary

The universe ranks instruments using average daily value, realized volatility, observation count and data quality checks. The tradable flag is a conservative diagnostic, not a recommendation.


In [ ]:
from __future__ import annotations

import json
from pathlib import Path
import sys

import pandas as pd
import plotly.express as px

PROJECT_ROOT = Path.cwd().resolve()
if not (PROJECT_ROOT / 'src').exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
sys.path.insert(0, str(PROJECT_ROOT / 'src'))

from russian_markets_lab.data.io import read_processed_dataset, read_processed_metadata

pd.set_option('display.max_columns', 80)
pd.set_option('display.width', 140)

def load_dataset(name: str) -> tuple[pd.DataFrame, dict]:
    df = read_processed_dataset(name)
    metadata = read_processed_metadata(name)
    print(f'{name}: {len(df)} rows, {len(df.columns)} columns')
    if metadata:
        print('generated_at:', metadata.get('generated_at'))
        print('source:', metadata.get('source'))
    else:
        print('metadata: missing')
    return df, metadata

def show_missing(name: str) -> None:
    print(f'No processed data found for {name}. Run the corresponding CLI pipeline first.')


In [ ]:
universe, metadata = load_dataset('market_universe')
if universe.empty:
    show_missing('market_universe')
else:
    display(universe.head(20))


## Top Instruments By Tradability And Liquidity


In [ ]:
if not universe.empty:
    columns = [c for c in ['ticker', 'name', 'board', 'avg_daily_value', 'realized_volatility', 'tradable_flag', 'tradability_score', 'data_quality_score'] if c in universe.columns]
    display(universe.sort_values(['tradable_flag', 'tradability_score'], ascending=[False, False])[columns].head(20))


## Volatility Versus Average Value


In [ ]:
if {'avg_daily_value', 'realized_volatility'}.issubset(universe.columns) and not universe.empty:
    fig = px.scatter(universe, x='avg_daily_value', y='realized_volatility', color='tradable_flag' if 'tradable_flag' in universe.columns else None, hover_name='ticker' if 'ticker' in universe.columns else None, log_x=True, title='Market universe: liquidity versus volatility')
    fig.show()
    try:
        fig.write_image(PROJECT_ROOT / 'assets' / 'dashboard_market_map.png')
    except Exception as exc:
        print('Chart export skipped:', exc)


## Key Observations

Use this section after execution to describe concentration, liquidity tiers, short histories and instruments excluded by data quality checks.


## Limitations And Disclaimer

Public ISS data may be delayed or incomplete. Tradability is a research diagnostic and not investment advice.
